# LangChain Serializable Reference

Developer-facing statements defined in `langchain_core.load.serializable`.

# `BaseSerialized: TypedDict`

Base structure shared by LangChain serialization records.

```python
lc: int # Serialization format version
id: list[str] # Unique object identifier
name: NotRequired[str] # Optional object name
graph: NotRequired[dict[str, Any]] # Optional object graph
```

---

# `SerializedConstructor: BaseSerialized`

Serialization record for reconstructing an object from constructor arguments.

```python
type: Literal["constructor"] # Record type
kwargs: dict[str, Any] # Constructor arguments
```

It also contains the fields inherited from `BaseSerialized`.

---

# `SerializedSecret: BaseSerialized`

Serialization record representing a secret reference.

```python
type: Literal["secret"] # Record type
```

It also contains the fields inherited from `BaseSerialized`.

---

# `SerializedNotImplemented: BaseSerialized`

Serialization record used when an object does not support LangChain serialization.

```python
type: Literal["not_implemented"] # Record type
repr: str | None # Object representation when available
```

It also contains the fields inherited from `BaseSerialized`.

---

# `try_neq_default`

Determines whether a Pydantic model field value differs from its declared default.

```python
try_neq_default(
    value: Any, # Field value to compare
    key: str, # Name of the model field
    model: BaseModel, # Pydantic model containing the field
) -> bool # Whether the value differs from the field default
```

The comparison handles values whose inequality result cannot be converted directly to `bool`.

---

# `Serializable: BaseModel, ABC`

Pydantic base model for opt-in LangChain serialization. It defines no abstract methods and is not serializable unless a subclass enables serialization.

## Constructor

```python
Serializable(
    *args: Any, # Positional arguments passed to BaseModel
    **kwargs: Any, # Keyword arguments passed to BaseModel
) -> None
```

## Properties

### `lc_secrets`

Returns constructor argument names mapped to secret identifiers.

```python
lc_secrets: dict[str, str]
```

The default is an empty dictionary. Subclasses can override this property to replace matching serialized values with secret records.

### `lc_attributes`

Returns additional constructor-compatible attributes to include in serialized arguments.

```python
lc_attributes: dict[str, Any]
```

The default is an empty dictionary.

## Methods

### `is_lc_serializable`

Reports whether the class supports LangChain serialization.

```python
@classmethod
is_lc_serializable(
    cls,
) -> bool # Whether the class supports serialization
```

The default is `False`.

### `get_lc_namespace`

Returns the namespace used to construct the serialization identifier.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # Namespace components
```

The default splits `cls.__module__` on `"."`. New partner packages should use this default; legacy overrides exist only for backwards-compatible identifiers.

### `lc_id`

Returns the class serialization identifier.

```python
@classmethod
lc_id(
    cls,
) -> list[str] # Namespace components followed by the original class name
```

For Pydantic generic models, the identifier uses the origin class name rather than the generated generic class name.

### `to_json`

Serializes the model into a LangChain constructor record when serialization is enabled.

```python
to_json(
    self,
) -> SerializedConstructor | SerializedNotImplemented # Serialized record
```

Fields that are not useful constructor arguments or are explicitly excluded are omitted. Attributes and secret mappings are merged across the subclass hierarchy, and secret values are replaced with `SerializedSecret` records.

Returns `to_json_not_implemented()` when `is_lc_serializable()` is `False`.

Raises `ValueError` when a class in the hierarchy defines the deprecated `lc_namespace` or `lc_serializable` attribute.

### `to_json_not_implemented`

Returns a not-implemented serialization record for the current object.

```python
to_json_not_implemented(
    self,
) -> SerializedNotImplemented # Not-implemented serialization record
```

This method delegates to the module-level `to_json_not_implemented()` function.

## Behaviour

Extra input fields are ignored through `ConfigDict(extra="ignore")`. Model representations omit fields whose values match their defaults.

---

# `to_json_not_implemented`

Creates a not-implemented serialization record for an object.

```python
to_json_not_implemented(
    obj: object, # Object that could not be serialized
) -> SerializedNotImplemented # Not-implemented serialization record
```

The record contains serialization version `1`, type `"not_implemented"`, the best available module-and-class identifier, and the object's `repr` when it can be obtained.

In [ ]:
from langchain_core.load.dump import dumps # Import the JSON serialization function
from langchain_core.load.serializable import Serializable, to_json_not_implemented # Import serialization tools


class AppConfig(Serializable): # Create a serializable LangChain model
    app_name: str # Store the application name
    api_key: str # Store a secret API key

    @classmethod
    def is_lc_serializable(cls) -> bool: # Enable LangChain serialization
        return True # Mark this class as serializable

    @property
    def lc_secrets(self) -> dict[str, str]: # Define which fields contain secrets
        return {"api_key": "APP_API_KEY"} # Replace the API key with a secret reference


config = AppConfig(app_name="Demo App", api_key="secret-123") # Create an example object

print("Class ID:", config.lc_id()) # Display the serialization identifier
print("Serialized dictionary:", config.to_json()) # Display the serialized dictionary
print("Serialized JSON:") # Print a heading
print(dumps(config, pretty=True)) # Display JSON without exposing the real API key

unsupported = to_json_not_implemented(100) # Serialize an unsupported object
print("Unsupported object:", unsupported) # Display the not-implemented record